# 08 · EDA y estadística con PySpark

**Tema 13 · Notebook guiado de Big Data.** Usa este cuaderno en paralelo con el [09 en pandas](09_eda_y_estadistica_bigdata.ipynb). Ambos siguen la misma encuesta, las mismas reglas de limpieza y los mismos contrastes. Ejecuta las celdas de arriba abajo. Cada paso explica qué queremos averiguar, cómo lo hacemos y cómo leer la salida. Trabajaremos con tablas y resultados numéricos.

**Pregunta de la demostración.** ¿Cuánto difieren los salarios mensuales medios de los grupos profesionales A y B? Después veremos una pregunta sobre categorías y otra sobre la relación entre dos medidas numéricas.

Los archivos `empleados_curso.csv` y `ciudades_contexto.csv` están en `PySpark/data`. Puedes iniciar el kernel desde la raíz del repositorio, `PySpark` o `PySpark/notebooks`. Sigue la [guía del entorno de PySpark](../01_environment_pyspark.md) para usar Java 17 y PySpark 3.5. También necesitas SciPy 1.11 o posterior para los contrastes. Si falta, ejecuta `%pip install "scipy>=1.11"` y reinicia el kernel si Jupyter lo solicita.

La encuesta es una versión reducida del sprint 8. Los nombres y DNI se retiraron. `id_persona` es un código local para reconocer filas de una misma persona. `id_registro` identifica copias exactas en el archivo original. Las tablas de clase usan resultados agregados.

## 1 · Entender qué tenemos

El análisis exploratorio de datos (EDA) nos ayuda a entender qué representa cada fila, detectar problemas y decidir si los datos permiten responder la pregunta. Aquí esperamos una fila por persona una vez revisadas las repeticiones.

Empezamos por las dimensiones, los tipos y los valores faltantes. Los porcentajes ayudan a comparar columnas con la misma base.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

inicio = Path.cwd()
archivos = ('empleados_curso.csv', 'ciudades_contexto.csv')
candidatas = (inicio / 'data', inicio.parent / 'data', inicio / 'PySpark' / 'data')
carpeta = next((ruta for ruta in candidatas
                if all((ruta / archivo).is_file() for archivo in archivos)), None)
if carpeta is None:
    raise FileNotFoundError('Abre el notebook desde el repositorio, PySpark o PySpark/notebooks.')

spark = SparkSession.builder.master('local[2]').appName('EDA de empleados').getOrCreate()

esquema = ('id_persona LONG, id_registro LONG, ciudad STRING, grupo STRING, '
           'salario_original STRING, tiene_coche STRING, km_anuales DOUBLE')
original = spark.read.csv((carpeta / archivos[0]).as_uri(), header=True, schema=esquema)

filas = original.count()
print(f'{filas} filas y {len(original.columns)} columnas')
original.drop('id_persona', 'id_registro').show(5, truncate=False)

nulos = original.agg(*[
    F.count(F.when(F.col(columna).isNull(), 1)).alias(columna)
    for columna in original.columns
]).first()
perfil = [(columna, tipo, nulos[columna], round(100 * nulos[columna] / filas, 1))
          for columna, tipo in original.dtypes]
spark.createDataFrame(perfil, ['columna', 'tipo', 'nulos', 'pct_nulos']).show(truncate=False)

**Lectura.** Un salario puede aparecer como texto por el símbolo de moneda o por los separadores. Un nulo en kilometraje también puede tener una explicación válida si la persona no tiene coche. Investigamos el motivo antes de rellenar valores.

## 2 · Evitar contar dos veces a una persona

Primero quitamos las copias exactas. Después distinguimos las filas sin identificador de las personas que conservan varias fichas diferentes. Para esta demostración apartamos todas las fichas conflictivas porque no tenemos una fecha fiable que indique cuál es la vigente. Guardamos el número de filas descartadas en cada paso.

In [ ]:
sin_copias = original.dropDuplicates(['id_registro'])
identificados = sin_copias.dropna(subset=['id_persona'])
repetidos = identificados.groupBy('id_persona').count().filter(F.col('count') > 1)
base = identificados.join(repetidos.select('id_persona'), 'id_persona', 'left_anti')

n_sin_copias = sin_copias.count()
n_identificados = identificados.count()
n_base = base.count()
auditoria = [
    ('filas de entrada', filas),
    ('copias exactas retiradas', filas - n_sin_copias),
    ('filas sin código de persona', n_sin_copias - n_identificados),
    ('filas con fichas conflictivas', n_identificados - n_base),
    ('filas para explorar', n_base),
]
spark.createDataFrame(auditoria, ['comprobacion', 'filas']).show(truncate=False)
assert base.select('id_persona').distinct().count() == n_base

## 3 · Pasar de texto a valores comparables

Corregimos grafías de ciudades presentes en el archivo y usamos los grupos definidos de A a D. En el salario retiramos marcas de moneda, interpretamos separadores según el patrón del archivo y traducimos tres importes escritos en catalán. Los valores ambiguos, vacíos o no positivos quedan sin importe utilizable.

En kilometraje, los negativos son inválidos. Si la persona declara que no tiene coche, el kilometraje queda como **no aplicable**, incluso si el archivo traía un número. Conservamos el original para auditar.

In [ ]:
grafias = {
    'ValÃ¨ncia': 'València', 'MÃ\xa0laga': 'Málaga', 'A CoruÃ±a': 'A Coruña',
    'DÃ¼sseldorf': 'Düsseldorf', 'MÃ¼nchen': 'München',
    'TromsÃ¸': 'Tromsø', 'KÃ¶ln': 'Köln',
}
grupos_validos = ['Grup A', 'Grup B', 'Grup C', 'Grup D']
base = (base.withColumn('ciudad', F.trim('ciudad'))
        .replace(grafias, subset=['ciudad'])
        .withColumn('ciudad', F.when(F.col('ciudad') != 'Desconeguda', F.col('ciudad')))
        .withColumn('grupo', F.trim('grupo'))
        .withColumn('grupo', F.when(F.col('grupo').isin(grupos_validos), F.col('grupo'))))

importe = F.trim(F.regexp_replace(F.lower('salario_original'), r'â‚¬|€|euros?', ''))
base = (base.withColumn('importe', importe)
        .replace({'mil vuit-cents': '1800', 'dos mil': '2000', 'tres mil': '3000'},
                 subset=['importe']))
# En este archivo, un separador seguido de tres cifras marca miles.
importe = F.regexp_replace('importe', r'(?<=\d)[.,](?=\d{3}(?:[.,]|$))', '')
importe = F.regexp_replace(importe, ',', '.')
salario = F.when(importe.rlike(r'^[+-]?\d+(?:\.\d{1,2})?$'), importe.cast('double'))

coche = F.lower('tiene_coche')
base = (base.withColumn('salario_eur', F.when(salario > 0, salario))
        .drop('importe')
        .withColumn('tiene_coche', F.when(coche == 'true', True).when(coche == 'false', False)))
sin_coche = F.coalesce(F.col('tiene_coche') == False, F.lit(False))
km_invalido = (F.col('km_anuales') < 0) | sin_coche
base = base.withColumn('km_limpios', F.when(~km_invalido, F.col('km_anuales')))

base.agg(
    F.count(F.when(F.col('grupo').isNull(), 1)).alias('grupo_invalido'),
    F.count(F.when(F.col('salario_eur').isNull(), 1)).alias('sin_salario'),
    F.count(F.when(sin_coche & F.col('km_anuales').isNotNull(), 1)).alias('km_sin_coche'),
).show()

## 4 · Describir antes de contrastar

Miramos cuántas personas hay por grupo, cuántas tienen salario utilizable y cómo se distribuyen esos importes. La media usa todos los valores. La mediana es el valor central. Los percentiles 25 y 75 delimitan la mitad central. La desviación típica resume cuánto se separan los importes de su media.

Compara la cobertura entre grupos. Si faltan más salarios en uno de ellos, el análisis de casos observados puede quedar sesgado.

In [ ]:
resumen = (base.dropna(subset=['grupo']).groupBy('grupo').agg(
    F.count('*').alias('personas'),
    F.count('salario_eur').alias('observados'),
    F.avg('salario_eur').alias('media'),
    F.percentile('salario_eur', 0.5).alias('mediana'),
    F.stddev_samp('salario_eur').alias('desviacion'),
    F.min('salario_eur').alias('minimo'),
    F.max('salario_eur').alias('maximo'),
    F.percentile('salario_eur', 0.25).alias('p25'),
    F.percentile('salario_eur', 0.75).alias('p75'),
).withColumn('pct_sin_salario', (1 - F.col('observados') / F.col('personas')) * 100))

resumen.orderBy('grupo').select(
    'grupo', 'personas', 'observados', 'mediana', 'minimo', 'maximo', 'p25', 'p75',
    F.round('media', 2).alias('media'),
    F.round('desviacion', 2).alias('desviacion'),
    F.round('pct_sin_salario', 2).alias('pct_sin_salario'),
).show()

## 5 · Nivel 2 breve. Añadir contexto por ciudad

La misma encuesta se puede enriquecer con un índice de coste de vida. El archivo de contexto contiene ciudades homónimas. Como la encuesta no incluye un país de residencia fiable para esta unión, apartamos las claves ambiguas. El país de origen de una persona no identifica dónde vive.

Usamos una unión izquierda para conservar todas las personas. Si falta un índice, dejamos ese valor vacío. Este contexto sirve para generar preguntas y queda fuera de los dos contrastes.

In [ ]:
import unicodedata

@F.udf('string')
def clave_ciudad(valor):
    if valor is None:
        return None
    valor = valor.replace(r'\u00f1', 'ñ')
    letras = unicodedata.normalize('NFKD', valor.strip().casefold())
    return ''.join(letra for letra in letras if not unicodedata.combining(letra))

contexto = spark.read.csv(
    (carpeta / archivos[1]).as_uri(), header=True,
    schema='ciudad STRING, pais_contexto STRING, indice_coste_vida DOUBLE',
)
contexto = (contexto.withColumn('clave_ciudad', clave_ciudad('ciudad'))
            .dropna(subset=['clave_ciudad']).dropDuplicates())
ambiguas = contexto.groupBy('clave_ciudad').count().filter(F.col('count') > 1).select('clave_ciudad')
contexto.join(ambiguas, 'clave_ciudad').select('ciudad', 'pais_contexto').orderBy('ciudad', 'pais_contexto').show()

contexto_unico = (contexto.join(ambiguas, 'clave_ciudad', 'left_anti')
                  .select('clave_ciudad', 'indice_coste_vida'))
ampliada = (base.withColumn('clave_ciudad', clave_ciudad('ciudad'))
            .join(contexto_unico, 'clave_ciudad', 'left'))
cobertura = ampliada.agg(
    F.count('*').alias('total'), F.count('indice_coste_vida').alias('con_contexto')
).first()
assert cobertura['total'] == n_base
print(f"Personas con contexto: {cobertura['con_contexto']} de {cobertura['total']}")

## 6 · Demostración 1. Comparar dos medias con Welch

**Pregunta.** ¿Difiere el salario medio mensual de los grupos A y B? Elegimos esos dos grupos para la demostración y calculamos A menos B.

**Hipótesis nula (H₀).** Las medias poblacionales son iguales. La alternativa contempla diferencias en ambos sentidos. Usamos un umbral del 5 % para ilustrar la decisión.

Welch admite varianzas diferentes. El contraste supone observaciones independientes. Verificamos una fila por persona y revisamos tamaños, extremos y dispersión con la tabla anterior. Una fila por persona evita el doble recuento, pero no demuestra independencia si hay, por ejemplo, personas agrupadas por equipos. Usamos salarios observados, sin sustituir ausencias por una media. La inferencia depende también del diseño de la encuesta. Spark calcula el tamaño, la media y la desviación típica muestral de cada grupo. Solo esos resúmenes pasan a SciPy para el contraste.

In [ ]:
from math import sqrt
from scipy import stats

alpha = 0.05
grupos_ab = ['Grup A', 'Grup B']
por_grupo = {fila['grupo']: fila for fila in
             resumen.filter(F.col('grupo').isin(grupos_ab))
             .select('grupo', 'observados', 'media', 'desviacion').collect()}
a, b = (por_grupo[grupo] for grupo in grupos_ab)
n_a, n_b = a['observados'], b['observados']

welch = stats.ttest_ind_from_stats(
    a['media'], a['desviacion'], n_a, b['media'], b['desviacion'], n_b,
    equal_var=False, alternative='two-sided',
)
diferencia = a['media'] - b['media']

# Intervalo de Welch a partir de las mismas estadísticas resumidas.
v_a, v_b = a['desviacion']**2 / n_a, b['desviacion']**2 / n_b
gl = (v_a + v_b)**2 / (v_a**2 / (n_a - 1) + v_b**2 / (n_b - 1))
margen = stats.t.ppf(1 - alpha / 2, df=gl) * sqrt(v_a + v_b)

print(f'Salarios observados: A = {n_a}, B = {n_b}')
print(f"Media A: {a['media']:.2f} € · Media B: {b['media']:.2f} €")
print(f'Diferencia A − B: {diferencia:.2f} €')
print(f'IC del 95 %: [{diferencia - margen:.2f}, {diferencia + margen:.2f}] €')
print(f'Valor p: {welch.pvalue:.4g}')
print('Decisión al 5 %:', 'rechazamos H₀' if welch.pvalue < alpha else 'no rechazamos H₀')

**Cómo leerlo.** Empieza por el signo y el tamaño de la diferencia en euros. Después mira el intervalo para valorar la precisión. El valor p resume lo inusual que sería observar una diferencia estandarizada al menos tan extrema bajo H₀ y los supuestos del test. Si no rechazamos H₀, la evidencia resulta insuficiente para distinguir las medias con este criterio.

La comparación es exploratoria. El cargo, la ubicación o la selección de quienes respondieron podrían influir en las cifras. La encuesta permite describir asociaciones y no atribuir causas.

## 7 · Demostración 2. Comparar categorías con chi-cuadrado

**Pregunta.** ¿Se relaciona el grupo profesional con declarar que se tiene coche? Usamos los grupos A y B y mostramos las proporciones sobre respuestas válidas.

**H₀.** Las dos variables son independientes. El test compara los recuentos observados con los esperados bajo H₀. Revisamos que las celdas tengan al menos cinco observaciones y cinco casos esperados como regla práctica para esta aproximación.

In [ ]:
recuentos = (base.filter(F.col('grupo').isin(grupos_ab)).groupBy('grupo').agg(
    F.count(F.when(F.col('tiene_coche') == False, 1)).alias('no'),
    F.count(F.when(F.col('tiene_coche') == True, 1)).alias('si'),
    F.count(F.when(F.col('tiene_coche').isNull(), 1)).alias('faltantes'),
).orderBy('grupo'))
recuentos.withColumn('pct_si', F.round(100 * F.col('si') / (F.col('si') + F.col('no')), 1)).show()

# Solo traemos dos filas agregadas al proceso de Python.
tabla = [[fila['no'], fila['si']] for fila in recuentos.collect()]
if len(tabla) != 2 or any(sum(fila) == 0 for fila in tabla) or any(sum(col) == 0 for col in zip(*tabla)):
    print('La tabla necesita dos grupos y dos respuestas para este contraste.')
else:
    chi = stats.chi2_contingency(tabla, correction=False)
    print('Esperados (columnas No, Sí):')
    print(chi.expected_freq.round(1))
    if min(min(fila) for fila in tabla) >= 5 and chi.expected_freq.min() >= 5:
        brecha = tabla[0][1] / sum(tabla[0]) - tabla[1][1] / sum(tabla[1])
        print(f'Diferencia A − B: {100 * brecha:.1f} puntos porcentuales')
        print(f'Chi-cuadrado: {chi.statistic:.3f} · Valor p: {chi.pvalue:.4g}')
        print('Decisión al 5 %:', 'rechazamos H₀' if chi.pvalue < alpha else 'no rechazamos H₀')
    else:
        print('La aproximación puede fallar con recuentos pequeños.')

**Cómo leerlo.** Mira qué proporción declaró tener coche en cada grupo y cuántas respuestas faltan. El test evalúa la asociación bajo sus supuestos. Las proporciones ayudan a valorar su tamaño. Los contrastes del cuaderno tienen un fin didáctico. Si quisiéramos confirmar varias hipótesis, planificaríamos cómo controlar los falsos positivos.

## 8 · Demostración 3. ¿Cambian juntas dos medidas?

**Pregunta.** Entre las personas que declaran tener coche, ¿quienes ganan más tienden a recorrer más kilómetros al año?

La **correlación** resume la dirección y la intensidad de una relación. El coeficiente va de −1 a +1. Si es positivo, valores altos de una variable tienden a acompañar valores altos de la otra. Si es negativo, tienden a acompañar valores bajos.

- **Pearson** mide en qué grado la relación sigue una línea recta. Los valores extremos pueden influir en el resultado.
- **Spearman** ordena cada variable de menor a mayor y compara esas posiciones. Puede captar una tendencia que sube o baja de forma consistente aunque no siga una recta.

Calculamos ambos sobre las mismas personas. El salario es el importe observado y el kilometraje es el que quedó tras la revisión. Si falta uno de los dos, la fila queda fuera de esta comparación. Para Pearson también hacemos un contraste bilateral. H₀ plantea una correlación lineal poblacional igual a cero. El valor p clásico requiere pares independientes y un modelo de normalidad conjunta bajo H₀. Usaremos el 5 % como umbral didáctico. Spearman se interpreta aquí como un coeficiente descriptivo.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

con_coche = base.filter(F.col('tiene_coche'))
pares = con_coche.select('salario_eur', 'km_limpios').dropna()
n_coche = con_coche.count()
corr = pares.agg(
    F.count('*').alias('n'),
    F.corr('salario_eur', 'km_limpios').alias('pearson'),
).first()
n, pearson = corr['n'], corr['pearson']

print(f'Personas que declaran tener coche: {n_coche}')
print(f'Pares completos: {n} · Sin algún dato: {n_coche - n}')
if n < 2 or pearson is None:
    print('Para calcular la correlación hacen falta pares completos y variación en ambas medidas.')
else:
    # La API de Spearman de Spark recibe las dos medidas juntas en un vector.
    vector = VectorAssembler(inputCols=['salario_eur', 'km_limpios'], outputCol='medidas')
    medidas = vector.transform(pares).select('medidas').cache()
    try:
        matriz = Correlation.corr(medidas, 'medidas', 'spearman').first()[0]
        spearman = matriz[0, 1]
    finally:
        medidas.unpersist()
    print(f'Pearson: {pearson:.3f} · Spearman: {spearman:.3f}')
    if n >= 3:
        # Contraste bilateral usando solo el coeficiente y el número de pares.
        r = abs(pearson)
        estadistico = r * sqrt((n - 2) / (1 - r**2)) if r < 1 else float('inf')
        p = 2 * stats.t.sf(estadistico, df=n - 2)
        print(f'Valor p bilateral de Pearson: {p:.4g}')
        print('Decisión al 5 %:', 'rechazamos H₀' if p < alpha else 'no rechazamos H₀')

**Cómo leerlo.** De 323 personas que declaran tener coche, tenemos 298 pares completos. Pearson da aproximadamente **−0,096** y Spearman **−0,070**. En esas filas, la relación lineal y la tendencia por orden son débiles. El valor p bilateral de Pearson ronda **0,099**. Con el umbral del 5 %, no rechazamos la hipótesis de correlación lineal cero bajo el modelo del contraste.

Un coeficiente cercano a cero puede ocultar una relación que cambia de dirección o diferencias entre grupos. También podrían influir la ciudad, el tipo de trabajo o quién dejó datos sin responder. Esta comparación es exploratoria. Por sí sola no permite concluir que el salario cambie los kilómetros recorridos.

[Pearson en Spark SQL](https://spark.apache.org/docs/3.5.6/api/python/reference/pyspark.sql/api/pyspark.sql.functions.corr.html) · [Spearman en Spark ML](https://spark.apache.org/docs/3.5.6/api/python/reference/api/pyspark.ml.stat.Correlation.html) · [Contraste de Pearson](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html) · [Spearman](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html)

## 9 · Comparar los resultados

Contrasta cada salida con el [09 en pandas](09_eda_y_estadistica_bigdata.ipynb). Esperamos los mismos recuentos y estimaciones, con posibles diferencias de redondeo. Las filas individuales permanecen en el DataFrame de Spark. Para SciPy usamos resúmenes de salarios, recuentos de categorías y el coeficiente de Pearson con su número de pares.

**Fuentes.** Adaptación del nivel 1 y de la unión por ciudad del nivel 2 de `sprint_8.ipynb`. [NIST, análisis exploratorio](https://itl.nist.gov/div898/handbook/eda/section1/eda11.htm). [SciPy, Welch](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html). [SciPy, chi-cuadrado](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.chi2_contingency.html).

[Volver al README](../README.md) · [Siguiente: EDA y estadística con pandas](09_eda_y_estadistica_bigdata.ipynb)

In [ ]:
spark.stop()